In [ ]:
!pip install sentence-transformers
!pip install faiss-cpu
!pip install PyPDF2
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 16.0 MB/s eta 0:00:00


In [ ]:
import os
import faiss
import numpy as np

from PyPDF2 import PdfReader

from sentence_transformers import SentenceTransformer

In [ ]:
import os

pdf_files = [f for f in os.listdir() if f.endswith(".pdf")]

print(pdf_files)

['RAG PAPER - 1.pdf', 'RAG PAPER -7.pdf', 'RAG PAPER-3.pdf', 'RAG PAPER -5.pdf', 'RAG PAPER-6.pdf', 'RAG PAPER -4.pdf', 'RAG PAPER - 2.pdf']


In [ ]:
documents = []

for pdf in pdf_files:

    reader = PdfReader(pdf)

    text = ""

    for page in reader.pages:

        extracted = page.extract_text()

        if extracted:
            text += extracted

    documents.append({
        "source": pdf,
        "text": text
    })

print("Documents Loaded:", len(documents))

Documents Loaded: 7


In [ ]:
chunks = []

chunk_size = 500

for doc in documents:

    text = doc["text"]

    for i in range(0, len(text), chunk_size):

        chunk = text[i:i+chunk_size]

        chunks.append({
            "source": doc["source"],
            "text": chunk
        })

print("Total Chunks:", len(chunks))

Total Chunks: 1579


In [ ]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding Model Loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


In [ ]:
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

embeddings = model.encode(
    chunk_texts,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

(1579, 384)


In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    np.array(embeddings)
)

print(
    "Vectors Stored:",
    index.ntotal
)

Vectors Stored: 1579


In [ ]:
def search(query, top_k=3):

    query_embedding = model.encode(
        [query]
    )

    distances, indices = index.search(
        np.array(query_embedding),
        top_k
    )

    results = []

    for idx in indices[0]:

        results.append(
            chunks[idx]
        )

    return results

In [ ]:
query = "What problem does RAG solve?"

results = search(query)

for r in results:

    print("\nSOURCE:", r["source"])
    print(r["text"][:500])


SOURCE: RAG PAPER -4.pdf
en MS-MARCO NLG by 2.6 Bleu
points and 2.6 Rouge-L points. RAG approaches state-of-the-art model performance, which is
impressive given that (i) those models access gold passages with speciﬁc information required to
generate the reference answer , (ii) many questions are unanswerable without the gold passages, and
(iii) not all questions are answerable from Wikipedia alone. Table 3 shows some generated answers
from our models. Qualitatively, we ﬁnd that RAG models hallucinate less and generate f

SOURCE: RAG PAPER -4.pdf
mise in being applied to a
wide variety of NLP tasks.
9Broader Impact
This work offers several positive societal beneﬁts over previous work: the fact that it is more
strongly grounded in real factual knowledge (in this case Wikipedia) makes it “hallucinate” less
with generations that are more factual, and offers more control and interpretability. RAG could be
employed in a wide variety of scenarios with direct beneﬁt to society, for example by

In [ ]:
query = "Difference between BERT and GPT"

results = search(query)

answer = ""

for r in results:

    answer += r["text"] + "\n"

print(answer[:3000])

onally made to make it as close to
GPT as possible so that the two methods could be
minimally compared. The core argument of this
work is that the bi-directionality and the two pre-
training tasks presented in Section 3.1 account for
the majority of the empirical improvements, but
we do note that there are several other differences
between how BERT and GPT were trained:
• GPT is trained on the BooksCorpus (800M
words); BERT is trained on the BooksCor-
pus (800M words) and Wikipedia (2,500M
words
odel sizes:
BERT BASE (L=12, H=768, A=12, Total Param-
eters=110M) and BERT LARGE (L=24, H=1024,
A=16, Total Parameters=340M).
BERT BASE was chosen to have the same model
size as OpenAI GPT for comparison purposes.
Critically, however, the BERT Transformer uses
bidirectional self-attention, while the GPT Trans-
former uses constrained self-attention where every
token can only attend to context to its left.4
1https://github.com/tensorﬂow/tensor2tensor
2http://nlp.seas.harvard.edu/2018/04/03/atte

In [ ]:
search("How does self attention work?")

[{'source': 'RAG PAPER - 1.pdf',
  'text': 'averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as\ndescribed in section 3.2.\nSelf-attention, sometimes called intra-attention is an attention mechanism relating different positions\nof a single sequence in order to compute a representation of the sequence. Self-attention has been\nused successfully in a variety of tasks including reading comprehension, abstractive summarization,\ntextual entailment and learning task-independent sentence representations [4, '},
 {'source': 'RAG PAPER - 1.pdf',
  'text': 'expensive than\nrecurrent layers, by a factor of k. Separable convolutions [ 6], however, decrease the complexity\nconsiderably, to O(k·n·d+n·d2). Even with k=n, however, the complexity of a separable\nconvolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,\nthe approach we take in our model.\nAs side benefit, self-attention could yield more interpretabl

In [ ]:
search("What problem does RAG solve?")

[{'source': 'RAG PAPER -4.pdf',
  'text': 'en MS-MARCO NLG by 2.6 Bleu\npoints and 2.6 Rouge-L points. RAG approaches state-of-the-art model performance, which is\nimpressive given that (i) those models access gold passages with speciﬁc information required to\ngenerate the reference answer , (ii) many questions are unanswerable without the gold passages, and\n(iii) not all questions are answerable from Wikipedia alone. Table 3 shows some generated answers\nfrom our models. Qualitatively, we ﬁnd that RAG models hallucinate less and generate f'},
 {'source': 'RAG PAPER -4.pdf',
  'text': 'mise in being applied to a\nwide variety of NLP tasks.\n9Broader Impact\nThis work offers several positive societal beneﬁts over previous work: the fact that it is more\nstrongly grounded in real factual knowledge (in this case Wikipedia) makes it “hallucinate” less\nwith generations that are more factual, and offers more control and interpretability. RAG could be\nemployed in a wide variety of scenari

In [ ]:
search("How does LoRA reduce training cost?")

[{'source': 'RAG PAPER-6.pdf',
  'text': ') LoRA can be combined with other efﬁcient adapta-\ntion methods, potentially providing orthogonal improvement. 2) The mechanism behind ﬁne-tuning\nor LoRA is far from clear – how are features learned during pre-training transformed to do well\non downstream tasks? We believe that LoRA makes it more tractable to answer this than full ﬁne-\n12tuning. 3) We mostly depend on heuristics to select the weight matrices to apply LoRA to. Are\nthere more principled ways to do it? 4) Finally, the rank-deﬁcienc'},
 {'source': 'RAG PAPER-6.pdf',
  'text': 'of Full Fine-tuning. A more general form of ﬁne-tuning allows the training of\na subset of the pre-trained parameters. LoRA takes a step further and does not require the accumu-\nlated gradient update to weight matrices to have full-rank during adaptation. This means that when\napplying LoRA to all weight matrices and training all biases2, we roughly recover the expressive-\nness of full ﬁne-tuning by se

In [ ]:
search("Difference between BERT and GPT")

[{'source': 'RAG PAPER - 2.pdf',
  'text': 'onally made to make it as close to\nGPT as possible so that the two methods could be\nminimally compared. The core argument of this\nwork is that the bi-directionality and the two pre-\ntraining tasks presented in Section 3.1 account for\nthe majority of the empirical improvements, but\nwe do note that there are several other differences\nbetween how BERT and GPT were trained:\n• GPT is trained on the BooksCorpus (800M\nwords); BERT is trained on the BooksCor-\npus (800M words) and Wikipedia (2,500M\nwords'},
 {'source': 'RAG PAPER - 2.pdf',
  'text': 'odel sizes:\nBERT BASE (L=12, H=768, A=12, Total Param-\neters=110M) and BERT LARGE (L=24, H=1024,\nA=16, Total Parameters=340M).\nBERT BASE was chosen to have the same model\nsize as OpenAI GPT for comparison purposes.\nCritically, however, the BERT Transformer uses\nbidirectional self-attention, while the GPT Trans-\nformer uses constrained self-attention where every\ntoken can only attend to

In [ ]:
def answer_question(query):

    results = search(query, top_k=3)

    print("="*60)
    print("QUESTION:", query)
    print("="*60)

    for i, r in enumerate(results):

        print(f"\nResult {i+1}")
        print("Source:", r["source"])
        print("-"*40)
        print(r["text"][:1000])

In [ ]:
answer_question("What is BERT?")

QUESTION: What is BERT?

Result 1
Source: RAG PAPER - 2.pdf
----------------------------------------
BERT and its detailed implementa-
tion in this section. There are two steps in our
framework: pre-training and ﬁne-tuning . Dur-
ing pre-training, the model is trained on unlabeled
data over different pre-training tasks. For ﬁne-
tuning, the BERT model is ﬁrst initialized with
the pre-trained parameters, and all of the param-
eters are ﬁne-tuned using labeled data from the
downstream tasks. Each downstream task has sep-
arate ﬁne-tuned models, even though they are ini-
tialized with the same pre

Result 2
Source: RAG PAPER - 2.pdf
----------------------------------------
-trained parameters. The
question-answering example in Figure 1 will serve
as a running example for this section.
A distinctive feature of BERT is its uniﬁed ar-
chitecture across different tasks. There is mini-mal difference between the pre-trained architec-
ture and the ﬁnal downstream architecture.
Model Architecture

In [ ]:
answer_question("What is self attention?")

QUESTION: What is self attention?

Result 1
Source: RAG PAPER - 1.pdf
----------------------------------------
averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as
described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions
of a single sequence in order to compute a representation of the sequence. Self-attention has been
used successfully in a variety of tasks including reading comprehension, abstractive summarization,
textual entailment and learning task-independent sentence representations [4, 

Result 2
Source: RAG PAPER - 1.pdf
----------------------------------------
ayers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
wise fully connected feed-forward network. We employ a residual connection [ 11] around each of
the two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is
LayerNorm( x+ Subla

In [ ]:
answer_question("How does LoRA reduce training cost?")

QUESTION: How does LoRA reduce training cost?

Result 1
Source: RAG PAPER-6.pdf
----------------------------------------
) LoRA can be combined with other efﬁcient adapta-
tion methods, potentially providing orthogonal improvement. 2) The mechanism behind ﬁne-tuning
or LoRA is far from clear – how are features learned during pre-training transformed to do well
on downstream tasks? We believe that LoRA makes it more tractable to answer this than full ﬁne-
12tuning. 3) We mostly depend on heuristics to select the weight matrices to apply LoRA to. Are
there more principled ways to do it? 4) Finally, the rank-deﬁcienc

Result 2
Source: RAG PAPER-6.pdf
----------------------------------------
of Full Fine-tuning. A more general form of ﬁne-tuning allows the training of
a subset of the pre-trained parameters. LoRA takes a step further and does not require the accumu-
lated gradient update to weight matrices to have full-rank during adaptation. This means that when
applying LoRA to all weight

In [ ]:
answer_question("What problem does RAG solve?")

QUESTION: What problem does RAG solve?

Result 1
Source: RAG PAPER -4.pdf
----------------------------------------
en MS-MARCO NLG by 2.6 Bleu
points and 2.6 Rouge-L points. RAG approaches state-of-the-art model performance, which is
impressive given that (i) those models access gold passages with speciﬁc information required to
generate the reference answer , (ii) many questions are unanswerable without the gold passages, and
(iii) not all questions are answerable from Wikipedia alone. Table 3 shows some generated answers
from our models. Qualitatively, we ﬁnd that RAG models hallucinate less and generate f

Result 2
Source: RAG PAPER -4.pdf
----------------------------------------
mise in being applied to a
wide variety of NLP tasks.
9Broader Impact
This work offers several positive societal beneﬁts over previous work: the fact that it is more
strongly grounded in real factual knowledge (in this case Wikipedia) makes it “hallucinate” less
with generations that are more factual, and o